# Seeded Run 1 — 4-Class Task × 10 Seeds (Colab GPU)

Trains **10 matched seeded pairs** for the 4-class SHD parity×language task on Google Colab.

**Key differences from the 5-seed local notebook:**
| Aspect | Local | Colab |
|--------|-------|-------|
| Seeds | 5 | **10** (more statistical power) |
| Optimizer | Adam | **AdamW** (decoupled weight decay) |
| Batch size | 256 | **512** (Colab T4/A100 has more VRAM) |
| Epochs | 25-50 | **50** |
| Checkpoints | Local disk | **Google Drive** (persistent) |

Cell 1 mounts Google Drive. Cell 2 sets up paths and verifies the environment.

## 0. Mount Google Drive & Install Dependencies

In [1]:
# ── Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Locate project root on Drive ──
import os
from pathlib import Path

# Try both "MyDrive" and "My Drive" (Colab uses either depending on account)
_DRIVE_BASE = "/content/drive/MyDrive"
if not os.path.exists(_DRIVE_BASE):
    _DRIVE_BASE = "/content/drive/My Drive"

# Look for the research project folder
_candidates = [
    f"{_DRIVE_BASE}/research project (SNN Info Theory)",
    f"{_DRIVE_BASE}/SNN Info Theory",
    f"{_DRIVE_BASE}/research project",
]

DRIVE_ROOT = None
for _c in _candidates:
    if os.path.exists(_c):
        DRIVE_ROOT = _c
        print(f"  ✓ Found project at: {DRIVE_ROOT}")
        break

if DRIVE_ROOT is None:
    # List Drive root to help user find correct path
    print(f"  ✗ Project folder not found in {_DRIVE_BASE}/")
    print("  Contents of Drive root:")
    for _item in os.listdir(_DRIVE_BASE):
        print(f"    - {_item}")
    raise FileNotFoundError(
        "Could not find 'research project (SNN Info Theory)' folder on Google Drive.\n"
        "Please upload the project folder to Google Drive and re-run."
    )

DRIVE_PROJECT = f"{DRIVE_ROOT}/Project Files"
DRIVE_DATA = f"{DRIVE_ROOT}/data/shd"
DRIVE_WIMFO = f"{DRIVE_ROOT}/wimfo"
DRIVE_PAPER = f"{DRIVE_ROOT}/neural_heterogeneity/SuGD_code"
DRIVE_TAU = f"{DRIVE_ROOT}/Tau from 15 epoch run.json"

# Verify critical files exist
_required = [
    ("seeded_runs_common.py", f"{DRIVE_PROJECT}/seeded_runs_common.py"),
    ("SHD train data", f"{DRIVE_DATA}/shd_train.h5"),
    ("SHD test data", f"{DRIVE_DATA}/shd_test.h5"),
    ("Tau artifact", DRIVE_TAU),
    ("model.py", f"{DRIVE_PAPER}/model.py"),
    ("layers.py", f"{DRIVE_PAPER}/layers.py"),
]
_missing = []
for _name, _path in _required:
    if os.path.exists(_path):
        print(f"  ✓ {_name}")
    else:
        print(f"  ✗ {_name} MISSING: {_path}")
        _missing.append(_name)

if _missing:
    raise FileNotFoundError(f"Missing required files: {', '.join(_missing)}")

# Install any missing packages
import importlib
for pkg in ["tables"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        print(f"  Installing {pkg}...")
        !pip install -q {pkg}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✓ Found project at: /content/drive/MyDrive/research project (SNN Info Theory)
  ✓ seeded_runs_common.py
  ✓ SHD train data
  ✓ SHD test data
  ✓ Tau artifact
  ✓ model.py
  ✓ layers.py


## 1. Imports & Path Setup

In [ ]:
import sys
sys.path.insert(0, DRIVE_PROJECT)
sys.path.insert(0, str(DRIVE_WIMFO))
sys.path.insert(0, str(DRIVE_PAPER))

from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import torch
from IPython.display import display

import seeded_runs_common as seeded_runs_common

# ── Override paths for Colab ──
#    IMPORTANT: do NOT reload after this — reload resets paths to Windows.
#    Module functions reference attributes dynamically, so overriding them
#    here is sufficient for all downstream calls.

_COLAB_BASE = Path(DRIVE_ROOT)

seeded_runs_common.PROJECT_ROOT = _COLAB_BASE
seeded_runs_common.PROJECT_FILES = _COLAB_BASE / "Project Files"
seeded_runs_common.WIMFO_ROOT = _COLAB_BASE / "wimfo"
seeded_runs_common.PAPER_ROOT = _COLAB_BASE / "neural_heterogeneity" / "SuGD_code"
seeded_runs_common.CHECKPOINT_ROOT = seeded_runs_common.PROJECT_FILES / "Checkpoints" / "SeededRuns"
seeded_runs_common.TAU_ARTIFACT_PATH = _COLAB_BASE / "Tau from 15 epoch run.json"
seeded_runs_common.SHD_TRAIN = _COLAB_BASE / "data" / "shd" / "shd_train.h5"
seeded_runs_common.SHD_TEST = _COLAB_BASE / "data" / "shd" / "shd_test.h5"

# Re-bind local variables
CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

RUN_LABEL = "seeded_run_1"
TASK_KEY = "4class"
MEM_DISTRIBUTION_FAMILY = "lognormal"
MASTER_SEEDS = [101, 202, 210, 340, 440, 550, 660, 710, 820, 930]

RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
RESULT_STEM = RUN_DIR / f"{RUN_LABEL}_checkpoint_summary"
PAIR_STEM = RUN_DIR / f"{RUN_LABEL}_pair_summary"
PAIRED_ACC_CSV = RUN_DIR / f"{RUN_LABEL}_paired_accuracy.csv"

TASK_DEF = TASKS[TASK_KEY]

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'N/A'}")
if DEVICE.type == 'cuda':
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Run directory: {RUN_DIR}")
print(f"Master seeds ({len(MASTER_SEEDS)}): {MASTER_SEEDS}")
print(f"Task: {TASK_DEF['task_name']} ({TASK_DEF['nb_outputs']} outputs)")


Device: cuda
GPU: NVIDIA L4


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## 2. Verification

In [ ]:
assert TASK_KEY == "4class"
assert TASK_DEF["nb_outputs"] == 4
assert TASK_DEF["task_name"] == "4class_parity_language"

# Verify 4-class label map
expected_map = {}
for i in range(20):
    is_german = int(i >= 10)
    is_odd = int(i % 2 == 1)
    expected_map[i] = is_german * 2 + is_odd
assert TASK_DEF["task_label_map"] == expected_map

preview = build_seeded_pair(
    master_seed=MASTER_SEEDS[0],
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
preview_meta = preview["metadata"]

assert preview["hom_prms"]["nb_outputs"] == 4
assert preview["hetero_prms"]["nb_outputs"] == 4
assert preview_meta["linear_sync_verified"]
assert preview_meta["hom_hidden_tau_unique"] == 1
assert preview_meta["hetero_hidden_tau_unique"] > 1

sampling_rows = build_sampling_preview_rows(
    MASTER_SEEDS,
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
sampling_df = pd.DataFrame(sampling_rows).sort_values("pair_seed").reset_index(drop=True)
assert not sampling_df["sample_matches_previous"].any()

display(sampling_df)
print(f"✓ All {len(MASTER_SEEDS)} seeds verified — unique τ_m samples, linear sync OK.")

## 3. Load Caches & Apply Optimizations

In [ ]:
# ── Load SHD data into RAM ──
train_cache, test_cache = load_default_caches()

In [ ]:
# ── 4-Class Optimizations (AdamW + Speed) ──────────────────────────
#   1. AdamW: decoupled weight decay, better than Adam for regularization
#   2. weight_decay=2e-4: light regularization (verified on 2-class v2)
#   3. batch_size=512: Colab T4/A100 has enough VRAM
#   4. nb_epochs=50: 4-class needs more training than 2-class
#   5. lr=2e-3: balanced speed/stability
#   6. LR scheduler patience=5: reduce LR when plateaus
#   NOTE: no reload — paths were set in Cell 5, reload would reset them

import torch

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# Optimizer + regularization
seeded_runs_common.BASE_PRMS["optimizer"] = "adamw"
seeded_runs_common.BASE_PRMS["weight_decay"] = 2e-4

# Training schedule
seeded_runs_common.BASE_PRMS["batch_size"] = 512
seeded_runs_common.BASE_PRMS["nb_epochs"] = 50
seeded_runs_common.BASE_PRMS["lr"] = 2e-3
seeded_runs_common.BASE_PRMS["lr_ab"] = 2e-3
seeded_runs_common.BASE_PRMS["lr_patience"] = 5
seeded_runs_common.BASE_PRMS["lr_factor"] = 0.5
seeded_runs_common.BASE_PRMS["lr_min"] = 5e-5
seeded_runs_common.BASE_PRMS["compile_model"] = False

print("4-class Colab optimizations applied:")
print(f"  Optimizer:     AdamW (wd={seeded_runs_common.BASE_PRMS['weight_decay']})")
print(f"  Batch size:    {seeded_runs_common.BASE_PRMS['batch_size']}")
print(f"  Epochs:        {seeded_runs_common.BASE_PRMS['nb_epochs']}")
print(f"  LR:            {seeded_runs_common.BASE_PRMS['lr']:.0e}")
print(f"  LR scheduler:  ReduceLROnPlateau(patience=5)")
print(f"  cuDNN bench:   {torch.backends.cudnn.benchmark}")


## 4. Training Helpers

In [ ]:
CHECKPOINT_KEY_FIELDS = ["pair_seed", "pair_role"]
PAIR_KEY_FIELDS = ["pair_seed"]

def show_run_status():
    checkpoint_rows = read_manifest_rows(RESULT_STEM)
    pair_rows = read_manifest_rows(PAIR_STEM)
    checkpoint_df = pd.DataFrame(checkpoint_rows)
    pair_df = pd.DataFrame(pair_rows)
    if not checkpoint_df.empty:
        checkpoint_df = checkpoint_df.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    if not pair_df.empty:
        pair_df = pair_df.sort_values("pair_seed").reset_index(drop=True)
    paired_acc_df = pd.DataFrame()
    if not checkpoint_df.empty:
        paired_acc_df = (
            checkpoint_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
            .reset_index().sort_values("pair_seed").reset_index(drop=True)
        )
        if {"heterogeneous_sampled", "homogeneous_anchor"}.issubset(paired_acc_df.columns):
            paired_acc_df["hetero_minus_hom"] = (
                paired_acc_df["heterogeneous_sampled"] - paired_acc_df["homogeneous_anchor"]
            )
            paired_acc_df.to_csv(PAIRED_ACC_CSV, index=False)
    return pair_df, checkpoint_df, paired_acc_df

def train_one_seed(master_seed):
    run_rows, pair_meta = run_pair_training(
        master_seed=master_seed, train_cache=train_cache, test_cache=test_cache,
        task_key=TASK_KEY, mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
        run_label=RUN_LABEL, skip_existing=True,
    )
    checkpoint_rows = upsert_rows(read_manifest_rows(RESULT_STEM), run_rows, CHECKPOINT_KEY_FIELDS)
    pair_rows = upsert_rows(read_manifest_rows(PAIR_STEM), [build_pair_summary_row(pair_meta)], PAIR_KEY_FIELDS)
    write_manifest_rows(checkpoint_rows, RESULT_STEM)
    write_manifest_rows(pair_rows, PAIR_STEM)
    pair_df, checkpoint_df, paired_acc_df = show_run_status()
    seed_df = checkpoint_df[checkpoint_df["pair_seed"] == master_seed].reset_index(drop=True)
    return seed_df, pair_df, checkpoint_df, paired_acc_df

print("Helpers ready. Chance level: 25.0%")
print(f"Seeds to train: {len(MASTER_SEEDS)}")

## 5. Train All 10 Seeds

Each cell trains one seed pair (homo + hetero). Already-completed seeds are reused from Drive checkpoints.

In [ ]:
# Seed 101
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(101)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 202
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(202)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 210
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(210)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 340
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(340)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 440
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(440)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 550
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(550)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 660
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(660)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 710
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(710)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 820
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(820)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

In [ ]:
# Seed 930
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(930)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])

## 6. Results & Statistical Tests

In [ ]:
# ── Summary ──
pair_df, checkpoint_df, paired_acc_df = show_run_status()

if not checkpoint_df.empty:
    display(checkpoint_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss"]])

if not paired_acc_df.empty:
    display(paired_acc_df)

    from scipy import stats

    diffs = paired_acc_df["hetero_minus_hom"].values
    homo = paired_acc_df["homogeneous_anchor"].values
    hetero = paired_acc_df["heterogeneous_sampled"].values
    n = len(diffs)
    mean_diff = np.mean(diffs)
    sd_diff = np.std(diffs, ddof=1)

    print(f"\n{'='*60}")
    print(f"  4-Class × {n} Seeds — Statistical Results")
    print(f"{'='*60}")
    print(f"  Hetero > Homo: {int(np.sum(diffs > 0))}/{n}")
    print(f"  Mean Δacc:     {mean_diff*100:+.2f} pp")
    print(f"  SD Δacc:       {sd_diff*100:.2f} pp")

    if n >= 2:
        t_stat, p_ttest = stats.ttest_rel(hetero, homo, alternative="greater")
        w_stat, p_wilcoxon = stats.wilcoxon(hetero, homo, alternative="greater")
        cohens_d = mean_diff / sd_diff if sd_diff > 0 else 0.0
        ci = stats.t.interval(0.95, df=n-1, loc=mean_diff, scale=sd_diff/np.sqrt(n))

        print(f"  95% CI:        [{ci[0]*100:+.2f}, {ci[1]*100:+.2f}] pp")
        print(f"  Cohen's d:     {cohens_d:.3f}")
        print(f"  t-test p:      {p_ttest:.6f}  {'★' if p_ttest < 0.05 else '—'} {'significant' if p_ttest < 0.05 else ''}")
        print(f"  Wilcoxon p:    {p_wilcoxon:.6f}  {'★' if p_wilcoxon < 0.05 else '—'} {'significant' if p_wilcoxon < 0.05 else ''}")

        print(f"\n  ── Perez-Nieves comparison ──")
        print(f"  Benchmark:     +3–8 pp (20-class SHD)")
        if mean_diff >= 0.03:
            print(f"  Verdict:       ✓ Within/above Perez-Nieves range")
        elif mean_diff >= 0.02:
            print(f"  Verdict:       ~ Close to Perez-Nieves range")
        elif mean_diff > 0:
            print(f"  Verdict:       — Positive but below benchmark")
        else:
            print(f"  Verdict:       ✗ No heterogeneity advantage")

    print(f"\n  Checkpoints saved to: {RUN_DIR}")